In [0]:
# STEP 1 Load dataset into Delta Table

#READ CSV
df_master_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/dataset/customer_master.csv")
)

display(df_master_raw)

customer_id,name,email,city,status,updated_at
1,Ravi Sharma,ravi.sharma@example.com,Jaipur,active,2026-01-01
2,Anjali Verma,anjali.verma@example.com,Delhi,active,2026-01-01
3,Suresh Kumar,suresh.kumar@example.com,Mumbai,active,2026-01-01
4,Priya Singh,priya.singh@example.com,Bangalore,active,2026-01-01
5,Amit Joshi,amit.joshi@example.com,Pune,active,2026-01-01
6,Neha Gupta,neha.gupta@example.com,Delhi,active,2026-01-01
7,Rahul Mehta,rahul.mehta@example.com,Chennai,active,2026-01-01
8,Kiran Rao,kiran.rao@example.com,Hyderabad,active,2026-01-01
9,Deepak Nair,deepak.nair@example.com,Kochi,active,2026-01-01
10,Deepak Nair,deepak.nair@example.com,Kochi,active,2026-01-01


In [0]:
#Total Rows
print(df_master_raw.count())

12


In [0]:
#Save as Delta Table
(
    df_master_raw.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("delta_table")
)

In [0]:
display(spark.table("delta_table"))

customer_id,name,email,city,status,updated_at
1,Ravi Sharma,ravi.sharma@example.com,Jaipur,active,2026-01-01
2,Anjali Verma,anjali.verma@example.com,Delhi,active,2026-01-01
3,Suresh Kumar,suresh.kumar@example.com,Mumbai,active,2026-01-01
4,Priya Singh,priya.singh@example.com,Bangalore,active,2026-01-01
5,Amit Joshi,amit.joshi@example.com,Pune,active,2026-01-01
6,Neha Gupta,neha.gupta@example.com,Delhi,active,2026-01-01
7,Rahul Mehta,rahul.mehta@example.com,Chennai,active,2026-01-01
8,Kiran Rao,kiran.rao@example.com,Hyderabad,active,2026-01-01
9,Deepak Nair,deepak.nair@example.com,Kochi,active,2026-01-01
10,Deepak Nair,deepak.nair@example.com,Kochi,active,2026-01-01


In [0]:
#Total Rows In Delta Table
print("Rows :", spark.table("delta_table").count())

Rows : 12


In [0]:
#STEP 2 Data Cleaning

#Remove NULL values

from pyspark.sql.functions import col
df_clean = spark.table("delta_table").dropna()

In [0]:
#Remove Duplicates
df_clean = df_clean.dropDuplicates()

In [0]:
# Cleaned data is saved in Delta Table
(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("delta_table")
)



In [0]:
#After removing NULL and Duplicate Values
print(df_clean.count())

10


In [0]:
#STEP 3  LOAD INCREMENTAL CSV AND READING IT
df_incremental = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/dataset/customer_incremental.csv")
)

display(df_incremental)

customer_id,name,email,city,status,updated_at
1,Ravi Sharma,ravi.sharma@example.com,Udaipur,active,2026-02-01
3,Suresh Kumar,suresh.kumar@example.com,Mumbai,inactive,2026-02-01
6,Neha Gupta,neha.gupta.new@example.com,Delhi,active,2026-02-02
11,Meera Iyer,meera.iyer@example.com,Chennai,active,2026-02-02
12,Arjun Nair,arjun.nair@example.com,Kochi,active,2026-02-03
8,Kiran Rao,kiran.rao@example.com,Hyderabad,active,2026-02-03


In [0]:
print(df_incremental.count())

6


In [0]:
#STEP 4 MERGE OPERATION 
from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark, "delta_table")

In [0]:
(
    delta_table.alias("target")
    .merge(
        df_incremental.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
).show()

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|                6|               4|               0|                2|
+-----------------+----------------+----------------+-----------------+



In [0]:
#STEP 5
#FINAL ROW COUNT
print("Final Row Count:", spark.table("delta_table").count())

Final Row Count: 12


In [0]:
#DUPLICATES CHECK AFTER CLEANING
from pyspark.sql.functions import count

dup_df = spark.table("delta_table").groupBy("customer_id").count().filter("count > 1")
dup_count = dup_df.count()

if dup_count == 0:
    print("No duplicate customer_id values found.")
else:
    print(f"{dup_count} duplicate customer_id value(s) found.")
    dup_df.show()

No duplicate customer_id values found.


In [0]:
#STEP 6  DISPLAY FINAL DATASET
display(spark.table("delta_table"))

customer_id,name,email,city,status,updated_at
2,Anjali Verma,anjali.verma@example.com,Delhi,active,2026-01-01
7,Rahul Mehta,rahul.mehta@example.com,Chennai,active,2026-01-01
10,Deepak Nair,deepak.nair@example.com,Kochi,active,2026-01-01
4,Priya Singh,priya.singh@example.com,Bangalore,active,2026-01-01
5,Amit Joshi,amit.joshi@example.com,Pune,active,2026-01-01
9,Deepak Nair,deepak.nair@example.com,Kochi,active,2026-01-01
1,Ravi Sharma,ravi.sharma@example.com,Udaipur,active,2026-02-01
3,Suresh Kumar,suresh.kumar@example.com,Mumbai,inactive,2026-02-01
6,Neha Gupta,neha.gupta.new@example.com,Delhi,active,2026-02-02
11,Meera Iyer,meera.iyer@example.com,Chennai,active,2026-02-02


In [0]:
# Summary of Week 7
1. Load - Raw customer data was loaded from CSV and saved as a Delta table.
2. Clean - Null values and duplicate rows were removed, and the cleaned data was saved back to the Delta table.
3. Incremental Data - A second CSV was used to simulate new/updated customer records arriving later.
4. Merge - Delta Lake's MERGE operation updated matching customers and inserted new ones based on customer_id.
5. Validate - Row count and duplicate checks confirmed the merge worked correctly with no duplicate customer_ids.
6. Final Output: The updated table was displayed.

Conclusion: Delta Lake's MERGE enables reliable incremental updates , updating existing records and inserting new ones in a single atomic operation.